# 04 - Error analysis

Where the models actually fail: weather regime, season, time of day,
generation level and ramping.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import pandas as pd
pd.set_option("display.width", 220)


In [ ]:
error = pd.read_csv(ROOT / "results" / "tables" / "error_analysis.csv")
intraday = error[error.stratum_type == "time_of_day"]
display(intraday[["model", "stratum", "n", "nrmse_capacity", "mae"]]
        .sort_values(["model", "nrmse_capacity"]).head(14))


In [ ]:
regime = pd.read_csv(ROOT / "results" / "tables" /
                        "weather_regime_comparison.csv")
display(regime.pivot_table(index="stratum", columns="model",
                            values="skill_vs_persistence_rmse").round(3))


Absolute error is **lowest** in cloudy conditions, because irradiance and
therefore the signal amplitude are small. Skill is also lowest there, which
is the opposite conclusion. Reading the error column alone inverts the
answer to the question a forecaster actually asks.


In [ ]:
display(regime[regime.stratum == "Cloudy"]
        [["model", "n", "nrmse_capacity", "skill_vs_persistence_rmse"]]
        .round(4))


In [ ]:
stats = pd.read_csv(ROOT / "results" / "tables" /
                      "statistical_significance.csv")
display(stats[stats.loss == "mae"]
        [["model", "skill_point", "skill_ci_low", "skill_ci_high",
          "dm_p_value", "dm_p_value_holm",
          "dm_lag1_autocorrelation"]].round(4))


## What to take from this notebook

- The block-bootstrap intervals and the Diebold-Mariano test disagree,
  because consecutive 15-minute errors share a cloud field. Both are
  reported, together with the autocorrelation that explains the
  disagreement.
- Stratifying by regime changes the conclusion about where forecasting is
  hard. That is the argument for stratified evaluation.
